In [2]:
print("Program started...")
"""
Hybrid Transformer-GARCH Framework for Financial Volatility Forecasting
===========================================================================
Thesis Title:
An Explainable Uncertainty-Aware Hybrid Transformer Framework for Financial
Volatility Forecasting and Tail-Risk Intelligence

Author: Mubaraq Temitope Balogun
Degree: MSc

Description:
------------
This implementation builds a hybrid volatility forecasting framework that:

1. Downloads multi-market financial data
2. Computes log returns
3. Fits a GARCH volatility model
4. Builds a Transformer neural network for sequence learning
5. Applies Monte Carlo Dropout for uncertainty estimation
6. Computes VaR and CVaR tail-risk metrics
7. Generates SHAP explainability analysis
8. Evaluates forecasting performance

Recommended Environment:
------------------------
Python 3.10+
PyTorch
CUDA-enabled GPU (optional)

Required Libraries:
-------------------
pip install pandas numpy yfinance arch scikit-learn matplotlib seaborn
pip install torch torchvision torchaudio
pip install shap scipy statsmodels plotly

===========================================================================
"""

# ================================================================
# IMPORT LIBRARIES
# ================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import shap

from arch import arch_model
from scipy.stats import norm

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ================================================================
# CONFIGURATION
# ================================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEQUENCE_LENGTH = 30
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 0.001
DROPOUT_RATE = 0.2
MC_SAMPLES = 50


# ================================================================
# DATA COLLECTION
# ================================================================

class MarketDataCollector:
    """
    Downloads financial market data from Yahoo Finance.
    """

    def __init__(self, tickers, start_date, end_date):
        self.tickers = tickers
        self.start_date = start_date
        self.end_date = end_date

    def download_data(self):
        data = {}

        for ticker in self.tickers:
            print(f"Downloading {ticker}...")

            df = yf.download(
                ticker,
                start=self.start_date,
                end=self.end_date,
                progress=False
            )

            data[ticker] = df

        return data


# ================================================================
# PREPROCESSING
# ================================================================

class FinancialPreprocessor:
    """
    Preprocesses financial time-series data.
    """

    def __init__(self):
        self.scaler = MinMaxScaler()

    @staticmethod
    def compute_log_returns(prices):
        """
        Compute logarithmic returns.

        r_t = ln(P_t / P_t-1)
        """

        returns = np.log(prices / prices.shift(1))
        return returns.dropna()

    def normalize(self, data):
        scaled = self.scaler.fit_transform(data.reshape(-1, 1))
        return scaled.flatten()

    @staticmethod
    def handle_missing_values(data):
        return data.fillna(method="ffill").fillna(method="bfill")


# ================================================================
# GARCH VOLATILITY ESTIMATION
# ================================================================

class GARCHVolatilityModel:
    """
    Fits a GARCH(1,1) volatility model.
    """

    def __init__(self):
        self.model = None
        self.results = None

    def fit(self, returns):
        self.model = arch_model(
            returns * 100,
            vol="Garch",
            p=1,
            q=1,
            dist="normal"
        )

        self.results = self.model.fit(disp="off")

        return self.results

    def get_conditional_volatility(self):
        return self.results.conditional_volatility


# ================================================================
# DATASET CREATION
# ================================================================

class VolatilityDataset(Dataset):
    """
    PyTorch Dataset for sequence modeling.
    """

    def __init__(self, features, targets, sequence_length):
        self.features = features
        self.targets = targets
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.features) - self.sequence_length

    def __getitem__(self, idx):
        x = self.features[idx:idx + self.sequence_length]
        y = self.targets[idx + self.sequence_length]

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32)
        )


# ================================================================
# POSITIONAL ENCODING
# ================================================================

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() *
            (-np.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


# ================================================================
# HYBRID TRANSFORMER MODEL
# ================================================================

class HybridTransformerGARCH(nn.Module):
    """
    Hybrid Transformer-GARCH architecture.
    """

    def __init__(
        self,
        input_dim=1,
        d_model=64,
        nhead=4,
        num_layers=2,
        dropout=0.2
    ):
        super().__init__()

        self.embedding = nn.Linear(input_dim, d_model)

        self.positional_encoding = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.dropout = nn.Dropout(dropout)

        self.fc1 = nn.Linear(d_model, 32)
        self.fc2 = nn.Linear(32, 1)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.embedding(x)

        x = self.positional_encoding(x)

        x = self.transformer(x)

        x = x[:, -1, :]

        x = self.dropout(x)

        x = self.relu(self.fc1(x))

        output = self.fc2(x)

        return output


# ================================================================
# TRAINING FUNCTION
# ================================================================


def train_model(model, train_loader, optimizer, criterion):
    model.train()

    total_loss = 0

    for features, targets in train_loader:
        features = features.to(DEVICE)
        targets = targets.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(features)

        loss = criterion(outputs.squeeze(), targets)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


# ================================================================
# PREDICTION FUNCTION
# ================================================================


def predict(model, loader):
    model.eval()

    predictions = []
    actuals = []

    with torch.no_grad():
        for features, targets in loader:
            features = features.to(DEVICE)

            outputs = model(features)

            predictions.extend(outputs.cpu().numpy().flatten())
            actuals.extend(targets.numpy().flatten())

    return np.array(predictions), np.array(actuals)


# ================================================================
# MONTE CARLO DROPOUT
# ================================================================


def mc_dropout_prediction(model, x, samples=50):
    """
    Monte Carlo Dropout for uncertainty estimation.
    """

    model.train()

    predictions = []

    for _ in range(samples):
        pred = model(x).detach().cpu().numpy()
        predictions.append(pred)

    predictions = np.array(predictions)

    mean_prediction = predictions.mean(axis=0)
    uncertainty = predictions.std(axis=0)

    return mean_prediction, uncertainty


# ================================================================
# TAIL RISK METRICS
# ================================================================


def compute_var(returns, confidence_level=0.95):
    """
    Compute Value-at-Risk.
    """

    return np.percentile(returns, (1 - confidence_level) * 100)



def compute_cvar(returns, confidence_level=0.95):
    """
    Compute Conditional Value-at-Risk.
    """

    var = compute_var(returns, confidence_level)

    return returns[returns <= var].mean()


# ================================================================
# EVALUATION METRICS
# ================================================================


def evaluate_performance(actuals, predictions):
    mse = mean_squared_error(actuals, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actuals, predictions)

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae
    }


# ================================================================
# VISUALIZATION
# ================================================================


def plot_predictions(actuals, predictions):
    plt.figure(figsize=(14, 6))

    plt.plot(actuals, label="Actual Volatility")
    plt.plot(predictions, label="Predicted Volatility")

    plt.title("Hybrid Transformer-GARCH Forecasting")
    plt.xlabel("Time")
    plt.ylabel("Volatility")

    plt.legend()
    plt.grid(True)

    plt.show()


# ================================================================
# SHAP EXPLAINABILITY
# ================================================================


def generate_shap_explanations(model, sample_data):
    """
    Generate SHAP explainability analysis.
    """

    model.eval()

    background = sample_data[:100]

    explainer = shap.DeepExplainer(model, background)

    shap_values = explainer.shap_values(sample_data[:10])

    shap.summary_plot(shap_values, sample_data[:10])


# ================================================================
# MAIN EXECUTION PIPELINE
# ================================================================


def main():

    # ============================================================
    # STEP 1: DOWNLOAD DATA
    # ============================================================

    tickers = [
        "^GSPC",      # S&P 500
        "EURUSD=X",  # EUR/USD
        "BTC-USD",   # Bitcoin
        "GC=F"        # Gold Futures
    ]

    collector = MarketDataCollector(
        tickers=tickers,
        start_date="2015-01-01",
        end_date="2025-01-01"
    )

    market_data = collector.download_data()


    # ============================================================
    # STEP 2: PREPROCESSING
    # ============================================================

    preprocessor = FinancialPreprocessor()

    all_returns = []

    for ticker, df in market_data.items():

        prices = df["Close"]

        prices = preprocessor.handle_missing_values(prices)

        returns = preprocessor.compute_log_returns(prices)

        all_returns.extend(returns.values)

    returns_array = np.array(all_returns)


    # ============================================================
    # STEP 3: GARCH MODEL
    # ============================================================

    garch = GARCHVolatilityModel()

    garch.fit(returns_array)

    volatility = garch.get_conditional_volatility()

    volatility = np.array(volatility)


    # ============================================================
    # STEP 4: NORMALIZATION
    # ============================================================

    normalized_volatility = preprocessor.normalize(volatility)


    # ============================================================
    # STEP 5: TRAIN-TEST SPLIT
    # ============================================================

    split_index = int(len(normalized_volatility) * 0.8)

    train_data = normalized_volatility[:split_index]
    test_data = normalized_volatility[split_index:]


    # ============================================================
    # STEP 6: DATASET CREATION
    # ============================================================

    train_dataset = VolatilityDataset(
        train_data.reshape(-1, 1),
        train_data,
        SEQUENCE_LENGTH
    )

    test_dataset = VolatilityDataset(
        test_data.reshape(-1, 1),
        test_data,
        SEQUENCE_LENGTH
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

Program started...
